## 1. Kiểm tra GPU

In [ ]:
# Kiểm tra GPU
!nvidia-smi

import tensorflow as tf
print("\n" + "="*50)
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")
print("="*50)

## 2. Mount Google Drive (Tùy chọn)

Nếu bạn muốn lưu model vào Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục lưu model
!mkdir -p '/content/drive/MyDrive/emotion-recognition'

## 3. Clone Repository

In [ ]:
# Clone repository từ GitHub
!git clone https://github.com/yourusername/face-emotion-recognition.git
%cd face-emotion-recognition

# HOẶC upload file từ máy tính
# from google.colab import files
# uploaded = files.upload()

## 4. Cài đặt Dependencies

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q kaggle
!pip install -q opencv-python-headless

# Kiểm tra phiên bản
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

## 5. Tải Dataset

**Cách 1:** Sử dụng Kaggle API

In [ ]:
# Upload kaggle.json từ máy tính
from google.colab import files
files.upload()  # Chọn file kaggle.json

# Cấu hình Kaggle
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Tải dataset FER-2013
!python src/download_data.py

**Cách 2:** Upload dataset thủ công

In [ ]:
# Upload file zip dataset
from google.colab import files
uploaded = files.upload()

# Giải nén
!unzip -q archive.zip -d data/

# Kiểm tra
!ls -la data/

## 6. Kiểm tra Dataset

In [ ]:
import os

# Đếm số ảnh
train_dir = 'data/train'
test_dir = 'data/test'

emotions = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

print("Dataset Statistics:")
print("=" * 50)

total_train = 0
total_test = 0

for emotion in emotions:
    train_count = len(os.listdir(os.path.join(train_dir, emotion)))
    test_count = len(os.listdir(os.path.join(test_dir, emotion)))
    total_train += train_count
    total_test += test_count
    print(f"{emotion:10s}: Train={train_count:5d}, Test={test_count:4d}")

print("=" * 50)
print(f"Total:      Train={total_train:5d}, Test={total_test:4d}")

## 7. Visualize Dataset

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

# Hiển thị mẫu ảnh từ mỗi cảm xúc
fig, axes = plt.subplots(1, 7, figsize=(20, 3))

for idx, emotion in enumerate(emotions):
    emotion_dir = os.path.join(train_dir, emotion)
    sample_img = os.listdir(emotion_dir)[0]
    img_path = os.path.join(emotion_dir, sample_img)
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    
    axes[idx].imshow(img, cmap='gray')
    axes[idx].set_title(emotion.upper())
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 8. Training Model

Bắt đầu train model với GPU!

In [ ]:
# Train model
!python src/train.py

# Hoặc train với ít epochs hơn để test nhanh:
# Sửa file train.py: EPOCHS = 30

## 9. Lưu Model

In [ ]:
# Copy model và training history về Google Drive
!cp models/emotion_model.h5 '/content/drive/MyDrive/emotion-recognition/'
!cp models/training_history.png '/content/drive/MyDrive/emotion-recognition/'

print("✓ Model đã được lưu vào Google Drive!")

# Hoặc tải về máy tính
from google.colab import files
files.download('models/emotion_model.h5')
files.download('models/training_history.png')

## 10. Đánh giá Model

In [ ]:
from keras.models import load_model
from keras.preprocessing.image import ImageDataGenerator

# Load model
model = load_model('models/emotion_model.h5')

# Prepare test data
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    'data/test',
    target_size=(48, 48),
    batch_size=64,
    color_mode='grayscale',
    class_mode='categorical',
    shuffle=False
)

# Evaluate
test_loss, test_acc = model.evaluate(test_generator)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

## 11. Test Prediction

In [ ]:
import numpy as np
from keras.preprocessing import image

# Test trên một ảnh
test_img_path = 'data/test/happy/im0.png'  # Thay đổi đường dẫn
img = image.load_img(test_img_path, target_size=(48, 48), color_mode='grayscale')
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array /= 255.0

# Predict
predictions = model.predict(img_array)
emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']
predicted_emotion = emotion_labels[np.argmax(predictions)]
confidence = np.max(predictions) * 100

# Display
plt.figure(figsize=(6, 6))
plt.imshow(img, cmap='gray')
plt.title(f"Predicted: {predicted_emotion} ({confidence:.2f}%)")
plt.axis('off')
plt.show()

# Show probabilities
print("\nProbabilities for each emotion:")
for i, emotion in enumerate(emotion_labels):
    print(f"{emotion:10s}: {predictions[0][i]*100:.2f}%")

## 12. Kết luận

✅ Model đã được train xong với GPU

✅ Model và training history đã được lưu

**Bước tiếp theo:**
1. Tải model về máy: `models/emotion_model.h5`
2. Test trên webcam: `python src/realtime_detection.py`
3. Chạy web app: `python app.py`

**Tips:**
- Colab free có giới hạn GPU ~12 giờ/ngày
- Nhớ lưu model vào Drive trước khi session hết hạn
- Có thể train thêm epochs bằng cách load model và tiếp tục train